# 09a — DABs: One Bundle, Many Environments (dev / test / prod)

**Format:** 👨‍🏫 **trainer-driven demo** — the trainer runs the CLI commands in a terminal, participants watch and run the verification cells.

**Goal:** show that a Declarative Automation Bundle (formerly Databricks Asset Bundles, DABs) deploys **the same code** to every environment, while **targets + variables** change only the values: names, folders, schemas, schedules, data-quality thresholds.

| Exam domain (May 2026 guide) | What this demo covers |
|---|---|
| **Implementing CI/CD (10%)** | `targets`, `variables`, `mode: development` vs `production`, `presets`, `--var`, `BUNDLE_VAR_*`, `validate` / `deploy` / `run` / `destroy` |
| Working with Lakeflow Jobs (16%) | job parameters and schedules defined in YAML, paused per environment |

> Bundle used: `materials/cicd_environments/` — a tiny job (`env_demo_job`) that records *which environment it runs in* into `<catalog>.<schema>.env_demo_runs`.

## 1. The problem: same code, different environments

| What changes between environments | dev | test | prod |
|---|---|---|---|
| Who deploys | the developer | CI pipeline | CI pipeline (service principal) |
| Where the files go (`root_path`) | your user folder | restricted folder | service-principal folder |
| Resource names | `[dev <you>] env_demo_job` | `[test] env_demo_job` | `env_demo_job` |
| Data location | `dev_env_demo` schema | `test_env_demo` | `prod_env_demo` |
| Schedule | paused | paused | **active** |
| Quality threshold (`min_rows`) | 1 | 100 | 1000 |

**Anti-pattern:** copy the job three times and edit it by hand. **Bundle pattern:** one YAML definition + one `targets:` block that sets the values.

## 2. Bundle anatomy

```
materials/cicd_environments/          <- bundle root (run CLI commands here)
├── databricks.yml                    <- bundle name, variables, targets (dev / test / prod)
├── resources/env_demo_job.yml        <- the job — uses ${var.*} everywhere, no hard-coded env values
└── src/show_environment.py           <- task notebook: reads job parameters, writes one row per run
```

**Three mechanisms work together:**

| Mechanism | Where | Example from this bundle |
|---|---|---|
| **Variables** | `variables:` + `${var.name}` in resources | `schema: ${var.schema}`, `pause_status: ${var.schedule_pause_status}` |
| **Target overrides** | `targets.<t>.variables` | `test: variables: { schema: test_env_demo, min_rows: "100" }` |
| **Deployment mode / presets** | `targets.<t>.mode`, `presets` | `mode: development` → `[dev <user>]` prefix + paused schedules; `presets: name_prefix: "[test] "` |

**Variable precedence** (highest wins): `--var="name=value"` on the CLI → `BUNDLE_VAR_name` environment variable → `targets.<t>.variables` → `default` in `variables:`.

In [ ]:
# Show the bundle definition (walk up from this notebook until the bundle folder is found)
import os

def find_up(rel):
    d = os.getcwd()
    while True:
        if os.path.exists(os.path.join(d, rel)):
            return os.path.join(d, rel)
        if os.path.dirname(d) == d:
            return None
        d = os.path.dirname(d)

bundle_root = find_up("materials/cicd_environments")
for f in ["databricks.yml", "resources/env_demo_job.yml"]:
    print(f"===== {f}")
    print(open(os.path.join(bundle_root, f)).read())

## 3. 👨‍🏫 Validate every target — see the resolved values

In a terminal, from `materials/cicd_environments/` (Databricks CLI, tested with **v1.16.1**; use your profile, e.g. `-p TRAINING`):

```bash
databricks bundle validate -t dev  --var="catalog=retailhub_trainer"
databricks bundle validate -t test --var="catalog=retailhub_trainer"
databricks bundle validate -t prod --var="catalog=retailhub_trainer"

# See the FULLY RESOLVED configuration of one target (what would be deployed)
databricks bundle validate -t prod --var="catalog=retailhub_trainer" -o json
```

**Result of the live run (2026-09-14):**

| Target | Job name | `root_path` | Schedule | Parameters |
|---|---|---|---|---|
| dev | `[dev krzysztof_burejza] env_demo_job` | `/Workspace/Users/<you>/.bundle/env_demo/dev` | PAUSED | `schema=dev_env_demo`, `min_rows=1` |
| test | `[test] env_demo_job` | `/Workspace/Users/<you>/.bundle/env_demo/test` | PAUSED | `schema=test_env_demo`, `min_rows=100` |
| prod | `env_demo_job` | `/Workspace/Users/<you>/.bundle/env_demo/prod` | **UNPAUSED** | `schema=prod_env_demo`, `min_rows=1000` |

> 💡 Talking point: the job YAML never mentions dev/test/prod. `validate -o json` is the fastest way to answer *"what exactly will land in prod?"* — before anything is deployed.

> ⚠️ If `root_path` points to `/Workspace/Shared/...`, `validate` warns that the folder is **writable by all workspace users**. Production bundles go to a restricted folder (typically the service principal's home).

## 4. 👨‍🏫 Override values at deploy time (no YAML edit)

```bash
# One-off override on the command line
databricks bundle validate -t dev --var="catalog=retailhub_trainer" --var="min_rows=5000" -o json | grep -A1 min_rows

# Same thing via environment variable — this is how CI/CD systems pass secrets / per-run values
export BUNDLE_VAR_min_rows=5000
databricks bundle validate -t dev --var="catalog=retailhub_trainer" -o json | grep -A1 min_rows
unset BUNDLE_VAR_min_rows
```

> 🎯 **Exam:** CLI `--var` beats `BUNDLE_VAR_*`, which beats `targets.<t>.variables`, which beats the variable `default`.

## 5. 👨‍🏫 Deploy two environments and run them

```bash
databricks bundle deploy -t dev  --var="catalog=retailhub_trainer"
databricks bundle deploy -t test --var="catalog=retailhub_trainer"

databricks bundle run -t dev  --var="catalog=retailhub_trainer" env_demo_job
databricks bundle run -t test --var="catalog=retailhub_trainer" env_demo_job
```

Then open **Jobs & Pipelines**: two jobs from one definition — `[dev <user>] env_demo_job` and `[test] env_demo_job`, each with its own parameters, tags and a paused schedule.

> 🚫 **Do not deploy `prod` in class** — its schedule is `UNPAUSED` (nightly run). In a real project prod is deployed by CI/CD, with `run_as` a service principal, usually into a separate workspace (`workspace.host`).

## 6. Everyone — verify what was deployed

The cells below use the Databricks SDK and SQL: no CLI needed. Run them after the trainer's deploy + run.

In [ ]:
%run ../../setup/00_setup

In [ ]:
# Every deployed copy of env_demo_job: name, schedule, parameters, tags
from databricks.sdk import WorkspaceClient

w = WorkspaceClient()
rows = []
for j in w.jobs.list(expand_tasks=False):
    name = j.settings.name if j.settings else ""
    if "env_demo_job" not in name:
        continue
    s = w.jobs.get(j.job_id).settings
    params = {p.name: p.default for p in (s.parameters or [])}
    rows.append((name,
                 s.schedule.pause_status.value if s.schedule and s.schedule.pause_status else None,
                 params.get("env_name"), params.get("schema"), params.get("min_rows"),
                 str(dict(s.tags or {}))))

if not rows:
    print("No env_demo_job deployed yet — ask the trainer to run `databricks bundle deploy -t dev` / `-t test`.")
else:
    display(spark.createDataFrame(rows, "job_name string, schedule string, env_name string, schema string, min_rows string, tags string"))

In [ ]:
# The data side: each environment wrote into ITS OWN schema (trainer's catalog)
DEMO_CATALOG = "retailhub_trainer"   # the catalog passed with --var="catalog=..." by the trainer

queries = []
for schema in ["dev_env_demo", "test_env_demo", "prod_env_demo"]:
    if spark.sql(f"SHOW SCHEMAS IN {DEMO_CATALOG} LIKE '{schema}'").count() and \
       spark.sql(f"SHOW TABLES IN {DEMO_CATALOG}.{schema} LIKE 'env_demo_runs'").count():
        queries.append(f"SELECT '{schema}' AS schema_name, * FROM {DEMO_CATALOG}.{schema}.env_demo_runs")

if queries:
    display(spark.sql(" UNION ALL ".join(queries) + " ORDER BY run_ts DESC"))
else:
    print(f"No env_demo_runs tables in {DEMO_CATALOG} yet — run the job in at least one target first.")

## 7. dev vs production mode — what the CLI does for you

| Behaviour | `mode: development` | `mode: production` |
|---|---|---|
| Resource name prefix | `[dev <user_name>]` automatically | none (unless `presets.name_prefix`) |
| Schedules / triggers | **paused** by default (an explicit `pause_status` on the resource overrides it — this bundle sets it via `${var.schedule_pause_status}`) | kept as declared |
| Pipelines | `development: true` | as declared |
| Default `root_path` | `/Workspace/Users/<you>/.bundle/<bundle>/<target>` | must be set explicitly; restricted folder recommended |
| Concurrent runs / locks | relaxed | enforced |
| Who should deploy | developer | CI/CD with a **service principal** (`run_as`) |

**How CI/CD uses this bundle** (e.g. GitHub Actions / Azure DevOps):
1. PR opened → `databricks bundle validate -t test` (+ unit tests)
2. Merge to `main` → `databricks bundle deploy -t test` → `databricks bundle run -t test env_demo_job` (integration test)
3. Release tag → `databricks bundle deploy -t prod` with the service principal's credentials (`DATABRICKS_HOST`, `DATABRICKS_CLIENT_ID`, `DATABRICKS_CLIENT_SECRET` as pipeline secrets)

> 🎯 **Exam tips:** *targets* select the environment (`-t`); *variables* carry the per-environment values; `mode: development` = prefixed names + paused schedules; production deploys should run as a service principal; `bundle destroy` removes everything a target deployed.

## 8. 👨‍🏫 Cleanup (after the demo)

```bash
databricks bundle destroy -t dev  --var="catalog=retailhub_trainer" --auto-approve
databricks bundle destroy -t test --var="catalog=retailhub_trainer" --auto-approve
```

`destroy` removes the jobs and the bundle files of that target. The `*_env_demo` schemas are data, not bundle resources: drop them with `DROP SCHEMA retailhub_trainer.dev_env_demo CASCADE` (and `test_env_demo`) if you want a clean catalog.

← [09 — CI/CD & Automation](09_cicd_and_automation.ipynb) | **[ README](../../../README.md)** | [10 — Governance & Security →](10_governance_and_security.ipynb)